# Tabular ML pipeline

A reusable scikit-learn workflow for messy tabular data: a ColumnTransformer that imputes and scales numeric columns and imputes and one-hot encodes categorical columns, optional LASSO feature selection, and a tuned multi-model comparison. This demo runs on the offline synthetic dataset; `scripts/compare_models.py --dataset adult` runs the same comparison on the public UCI Adult dataset.

In [1]:
from tabml import synthetic_tabular, train_test_split_frame, compare_models
from tabml.compare import results_table

data = synthetic_tabular(n_rows=600, seed=0)
print('missing cells:', int(data.X.isna().sum().sum()))
train, test = train_test_split_frame(data, seed=0)
results = compare_models(train, test, models=['logreg', 'random_forest', 'gradient_boosting'])
print(results_table(results))

missing cells: 61


model                   CV AUC  test AUC  test acc
--------------------------------------------------
logreg                  0.6883    0.6554    0.6067
gradient_boosting       0.6366    0.6096    0.5867
random_forest           0.5832    0.5663    0.5667


In [2]:
from tabml import build_preprocessor, lasso_selected_features
import numpy as np

prep = build_preprocessor(data.numeric, data.categorical)
X = prep.fit_transform(data.X)
X = np.asarray(X.todense()) if hasattr(X, 'todense') else np.asarray(X)
selected = lasso_selected_features(X, data.y, alpha=0.01)
print(f'LASSO kept {len(selected)} of {X.shape[1]} encoded features')

LASSO kept 5 of 8 encoded features
